<a href="https://colab.research.google.com/github/niro2486/Statistical-Learning-e22093/blob/main/Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

## Sample Answer

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Define the restricted physical domain for theta
theta = np.linspace(0.01, 1.0, 1000)

# Calculate the Beta(8, 1.5) prior density
alpha_prior = 8
beta_prior = 1.5
prior_density = stats.beta.pdf(theta, alpha_prior, beta_prior)

# Create the plot using Plotly
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theta,
    y=prior_density,
    mode='lines',
    name='Beta(8, 1.5) Prior',
    line=dict(color='blue', width=2),
    fill='tozeroy',
    fillcolor='rgba(0, 0, 255, 0.1)'
))

# Update layout for better visualization
fig.update_layout(
    title="Initial Prior Density: Structural Stiffness Efficiency",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    hovermode="x"
)

fig.show()

## 1. Prior Belief Boundaries

The initial prior is defined as a Beta distribution: $\Theta^{(0)} \sim \text{Beta}(8, 1.5)$.

The expected prior stiffness efficiency is calculated using the analytical mean of a Beta distribution, $E[\Theta] = \frac{\alpha}{\alpha + \beta}$:

$$E[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx 0.8421$$

**Why this prior is appropriate:**
The Beta distribution is strictly bounded on the interval $[0,1]$, which perfectly maps to the physical domain of the structural efficiency factor $\theta$ (where 1.0 is pristine and 0 is critical failure). By setting $\alpha = 8$ and $\beta = 1.5$ ($\alpha > \beta$), the probability density is heavily skewed to the right. This mathematically encodes the engineering assumption that a newly monitored, pre-inspected component is highly likely to be structurally healthy, while still reserving a small probability for manufacturing defects or pre-existing hidden damage.

## 2. Structural Likelihood Formulation

Given the physical model $y_k = \theta \cdot K_{nominal} \cdot e^{\epsilon_k}$ with $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$, we can take the natural logarithm of both sides:

$$\ln(y_k) = \ln(\theta) + \ln(K_{nominal}) + \epsilon_k$$

Because $\epsilon_k$ is normally distributed, $\ln(y_k)$ follows a normal distribution with mean $\mu = \ln(\theta \cdot K_{nominal})$ and variance $\sigma^2$. Using the change of variables principle to find the probability density function of $y_k$ yields the standard log-normal PDF.

The likelihood contribution of a single measurement $y_k$ given $\theta$ is:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln(y_k) - \ln(\theta \cdot K_{nominal}))^2}{2\sigma^2} \right)$$

Because sequential sensor measurements are assumed to be conditionally independent given the true degradation state $\theta$, the joint likelihood for the running history vector $y^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihoods:

$$L(y^{(k)} \mid \theta) = \prod_{i=1}^{k} \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( - \frac{(\ln(y_i) - \ln(\theta \cdot K_{nominal}))^2}{2\sigma^2} \right)$$

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

An exact closed-form analytical solution does not exist here because the Beta prior and Log-normal likelihood do not form a conjugate pair.

The Beta prior kernel is proportional to $\theta^{\alpha-1}(1-\theta)^{\beta-1}$, which relies on polynomial terms of $\theta$. Conversely, the Log-normal likelihood embeds $\theta$ inside a squared logarithmic term within an exponential function: $\exp(-(\ln(y) - \ln(\theta K))^2)$. When you multiply these two algebraic structures together via Bayes' theorem, the resulting expression cannot be factored or simplified into the functional form (kernel) of any known standard probability distribution.

Consequently, the recursive relationship for the posterior density at step $k$, up to a proportionality constant, is strictly evaluated point-by-point:

$$f^{(k)}(\theta \mid y^{(k)}) \propto f^{(k-1)}(\theta \mid y^{(k-1)}) \cdot \exp\left( - \frac{(\ln(y_k) - \ln(\theta \cdot K_{nominal}))^2}{2\sigma^2} \right)$$

*(Note: The $1 / (y_k \sigma \sqrt{2\pi})$ term is dropped in this recursive update because it acts as a constant with respect to $\theta$ and gets absorbed by the final normalization step).*

## 4. Running Point Estimates

Since we cannot define the posterior parametrically, we rely on numerical integration over the bounded domain $(0,1]$ to extract point estimates.

**The Running Posterior Mean (Bayes Estimator / MMSE):**
$$\hat{\theta}_{Bayes}^{(k)} = \int_0^1 \theta \cdot f^{(k)}(\theta \mid y^{(k)}) \, d\theta$$

**The Running Maximum A Posteriori (MAP):**
$$\hat{\theta}_{MAP}^{(k)} = \underset{\theta \in (0,1]}{\arg\max} \, f^{(k)}(\theta \mid y^{(k)})$$

## 5. Algorithmic Grid Approximation and Normalization

To implement this sequentially in software, follow this numerical procedure:

1. **Define the Grid**: Create a densely populated, linearly spaced array of $\theta$ values spanning from slightly above 0 to 1.0 (e.g., `theta_grid = np.linspace(0.01, 1.0, 1000)`). The lower bound avoids a mathematically undefined $\ln(0)$ error when evaluating the likelihood.
2. **Initialize the Prior**: Compute the Beta(8, 1.5) density across `theta_grid` to establish the initial density array $f^{(0)}$.
3. **Sequential Update**: When a new measurement $y_k$ arrives, evaluate the log-normal likelihood function across the entire `theta_grid`. Perform element-wise multiplication of this likelihood array with the current posterior array $f^{(k-1)}$.
4. **Trapezoidal Normalization**: The resulting array is unnormalized. Calculate the area under the discrete curve using `area = np.trapezoid(unnormalized_pdf, theta_grid)`. Divide every element in the unnormalized array by this `area` to secure a valid probability density function $f^{(k)}$ that integrates to 1.
5. **Extract Estimators**:
   * For the MAP, find the index of the maximum value in the array and return the corresponding `theta_grid` value.
   * For the Bayes mean, compute the integral of `theta_grid * f_k` using `np.trapezoid`.

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Initialization and Parameters
np.random.seed(42)
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_steps = 15

# Grid setup (avoiding exactly 0 for log functions)
theta_grid = np.linspace(0.01, 1.0, 1000)

# Simulate Sensor Stream (y_k)
# true_mu = ln(theta_true * K_nominal)
true_mu = np.log(theta_true * K_nominal)
measurements = np.random.lognormal(mean=true_mu, sigma=sigma, size=n_steps)

# 2. Track Estimators
posterior_pdfs = []
theta_map = []
theta_bayes = []
milestones = [0, 1, 2, 5, 10, 15]

# Initial Prior (Step 0)
prior_pdf = stats.beta.pdf(theta_grid, 8, 1.5)
posterior_pdfs.append(prior_pdf)
theta_map.append(theta_grid[np.argmax(prior_pdf)])
theta_bayes.append(np.trapezoid(theta_grid * prior_pdf, theta_grid))

current_pdf = prior_pdf.copy()

# Sequential Bayesian Update
for k, y_k in enumerate(measurements, start=1):
    # Likelihood formulation
    mu_grid = np.log(theta_grid * K_nominal)
    # Using the standard normal PDF structure for log-normal likelihood contribution
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(-0.5 * ((np.log(y_k) - mu_grid) / sigma) ** 2)

    # Point-wise multiplication (Grid update)
    unnormalized_pdf = current_pdf * likelihood

    # Normalization via trapezoidal rule
    area = np.trapezoid(unnormalized_pdf, theta_grid)
    current_pdf = unnormalized_pdf / area

    # Save milestones
    if k in milestones:
        posterior_pdfs.append(current_pdf.copy())

    # Track point estimates
    theta_map.append(theta_grid[np.argmax(current_pdf)])
    theta_bayes.append(np.trapezoid(theta_grid * current_pdf, theta_grid))


# 3. Visualize Curves & Timeline using Plotly
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Posterior Density Evolution",
                                    "Convergence of Point Estimators"))

# Plot A: Posterior Densities
colors = ['#d3d3d3', '#add8e6', '#87cefa', '#4169e1', '#0000cd', '#000080']
for i, k_val in enumerate(milestones):
    fig.add_trace(go.Scatter(x=theta_grid, y=posterior_pdfs[i],
                             mode='lines', name=f'Step {k_val}',
                             line=dict(color=colors[i], width=2)), row=1, col=1)

# Reference line for true theta in Plot A
fig.add_vline(x=theta_true, line_dash="dash", line_color="red",
              annotation_text="True θ", row=1, col=1)

# Plot B: Estimator Convergence
steps = np.arange(0, n_steps + 1)
fig.add_trace(go.Scatter(x=steps, y=theta_map, mode='lines+markers',
                         name='MAP Estimate', line=dict(color='blue')), row=1, col=2)
fig.add_trace(go.Scatter(x=steps, y=theta_bayes, mode='lines+markers',
                         name='Bayes Mean', line=dict(color='green')), row=1, col=2)

# Reference line for true theta in Plot B
fig.add_hline(y=theta_true, line_dash="dash", line_color="red",
              annotation_text="True θ (0.68)", row=1, col=2)

fig.update_layout(title_text="Bayesian Structural Health Monitoring",
                  template="plotly_white", height=500)
fig.update_xaxes(title_text="Stiffness Factor (θ)", row=1, col=1)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_xaxes(title_text="Inspection Step (k)", row=1, col=2)
fig.update_yaxes(title_text="Estimated θ", row=1, col=2)

fig.show()